In [1]:
import kagglehub
import pandas as pd  # nao vai usar :(
import numpy as np
import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, coalesce, lower, regexp_replace, trim, when, length, upper
from pyspark.sql.types import DoubleType

/usr/local/lib/python3.12/dist-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [33]:
spark.conf.set("spark.sql.ansi.enabled", False)

In [2]:
spark = SparkSession.builder.appName("roblox_games_data_analisys").getOrCreate()

# Leitura dataset

## Carregar datasets


In [3]:
'''
roblox-games-data - 1/22/2022 to 5/2/2022 *outdated???

info: Date,Active Users,Favorites,Total Visits,Date Created,Last Updated,Server Size,Genre,Title

# Download latest version
roblox_games_data_path = kagglehub.dataset_download("databitio/roblox-games-data")

print("Path to dataset files:", roblox_games_data_path)
'''

'\nroblox-games-data - 1/22/2022 to 5/2/2022 *outdated???\n\ninfo: Date,Active Users,Favorites,Total Visits,Date Created,Last Updated,Server Size,Genre,Title\n\n# Download latest version\nroblox_games_data_path = kagglehub.dataset_download("databitio/roblox-games-data")\n\nprint("Path to dataset files:", roblox_games_data_path)\n'

In [4]:
'''
roblox-games-dataset -  27/10/2024  *newest

info: Rank,Name,Active,Visits,Favorites,Likes,Deslikes,Rating
'''

# Download latest version
roblox_games_path = kagglehub.dataset_download("biggiefats/roblox-games-dataset")

print("Path to dataset files:", roblox_games_path)


Using Colab cache for faster access to the 'roblox-games-dataset' dataset.
Path to dataset files: /kaggle/input/roblox-games-dataset


In [5]:
'''
roblox-dataset -  11/5/2024

info: Title,Creator,AgeRecomendation,ActivePlayers,Favorites,Visits,VoiceChat,
      Camera,Created,Updated,ServerSize,Genre,Likes,Dislikes,GameLink,Datafetch
'''

# Download latest version
roblox_dataset_path = kagglehub.dataset_download("jansenccruz/roblox-dataset")

print("Path to dataset files:", roblox_dataset_path)

Using Colab cache for faster access to the 'roblox-dataset' dataset.
Path to dataset files: /kaggle/input/roblox-dataset


In [6]:
#Load dataframes

path=roblox_dataset_path+"/roblox_games_data.csv"
print(path)
df_old = spark.read.csv(path, header=True, inferSchema=True)

/kaggle/input/roblox-dataset/roblox_games_data.csv


In [7]:
df_old=df_old.withColumnRenamed("Favorites","Favourites")

In [8]:
df_old.show()

+--------------------+--------------------+-----------------+------+----------+-------+-------------+-------------+----------+----------+----------+---------------+-----+--------+--------------------+---------------+
|               Title|             Creator|AgeRecommendation|Active|Favourites| Visits|    VoiceChat|       Camera|   Created|   Updated|ServerSize|          Genre|Likes|Dislikes|            GameLink|    DateFetched|
+--------------------+--------------------+-----------------+------+----------+-------+-------------+-------------+----------+----------+----------+---------------+-----+--------+--------------------+---------------+
|Catalog Avatar Cr...|         @ItsMuneeeb|         All Ages| 19469|   3438844|  2.8B+|    Supported|    Supported|  7/4/2021| 11/5/2024|        24|       Shopping|  1M+|   124K+|https://www.roblo...|11/5/2024 20:24|
| [HALLOWEEN 🎃] T...|            CTStudio|         Ages 13+|  2709|   2911356|988.3M+|    Supported|Not Supported| 1/15/2021| 11/5/2

In [9]:
#df_old.printSchema()

In [10]:
path=roblox_games_path+"/roblox_games.csv"
print(path)
df_new = spark.read.csv(path, header=True, inferSchema=True)

/kaggle/input/roblox-games-dataset/roblox_games.csv


In [11]:
df_new.show()

+----+--------------------+-------+--------------+----------+---------+---------+------+
|Rank|                Name| Active|        Visits|Favourites|    Likes| Dislikes|Rating|
+----+--------------------+-------+--------------+----------+---------+---------+------+
|  #1|         Blox Fruits|483,372|41,346,317,182|13,574,097|8,521,670|  676,846| 92.64|
|  #2|     Brookhaven 🏡RP|474,141|55,635,148,446|22,117,653|6,108,763|  955,845| 86.47|
|  #3| Dress To Impress 💜|297,764| 3,876,511,994| 3,182,036|2,042,092|  188,403| 91.55|
|  #4|    PETS GO! ✨ [NEW]|172,411|   145,691,211|   199,254|  275,267|   20,140| 93.18|
|  #5|    Murder Mystery 2|159,531|18,310,453,247|19,306,585|8,001,198|  786,705| 91.05|
|  #6|[UPDATE 1] Anime ...|142,586|   534,044,793|   578,491|1,592,383|   52,159| 96.83|
|  #7|The Strongest Bat...|142,531| 8,747,773,201| 4,177,434|2,931,689|  565,313| 83.83|
|  #8|Pet Simulator 99! 🎃|131,088| 1,527,851,114| 1,479,726|2,586,908|  106,245| 96.05|
|  #9|          Adopt Me

In [12]:
df_new.printSchema()

root
 |-- Rank: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Active: string (nullable = true)
 |-- Visits: string (nullable = true)
 |-- Favourites: string (nullable = true)
 |-- Likes: string (nullable = true)
 |-- Dislikes: string (nullable = true)
 |-- Rating: double (nullable = true)



## Normalização dos nomes

In [13]:

EVENT_REGEX = (
    r"(?i)\b("
    r"new\s+update|big\s+update|mega\s+update|huge\s+update|balance\s+update|bug\s+fixes?|"
    r"update|upd|release|revamp|rework|patch|"
    r"halloween|christmas|xmas|winter|summer|spring|easter|valentine'?s?|thanksgiving|"
    r"black\s+friday|april\s+fools?|new\s+year|lunar\s+new\s+year|"
    r"free\s+ugc|ugc|limited\s+time|limited|new\s+codes?|codes?|code|rewards?|giveaway|"
    r"live\s+event|event|"
    r"new\s+map|new\s+world|new\s+area|new\s+island|new\s+pets?|new\s+eggs?|new\s+boss|"
    r"new\s+raid|new\s+weapons?|new\s+skins?|new\s+units?|new\s+fruit|new\s+quest|new\s+mode|"
    r"beta|alpha|demo|testing|early\s+access|open\s+beta|closed\s+beta"
    r")\b"
)

normalized_column_name="Normalized_name"

def criar_nome_normalizado(df, coluna_nome="Name"):
    return (
        df
        .withColumn(normalized_column_name, lower(col(coluna_nome)))

        # Remove emojis e símbolos estranhos
        .withColumn(
            normalized_column_name,
            regexp_replace(col(normalized_column_name), r"[^\w\s\-\:\|\!\(\)\[\]&']", " ")
        )

        # Remove blocos como [UPDATE], (HALLOWEEN), [NEW CODES]
        .withColumn(
            normalized_column_name,
            regexp_replace(
                col(normalized_column_name),
                r"(?i)[\[\(]\s*[^)\]]*(update|upd|event|codes?|ugc|halloween|christmas|xmas|easter|winter|summer|beta|alpha)[^)\]]*[\]\)]",
                " "
            )
        )

        # Remove palavras de evento/update
        .withColumn(
            normalized_column_name,
            regexp_replace(col(normalized_column_name), EVENT_REGEX, " ")
        )

        # Padroniza e limpa separadores
        .withColumn(normalized_column_name, regexp_replace(col(normalized_column_name), r"&", " and "))
        .withColumn(normalized_column_name, regexp_replace(col(normalized_column_name), r"[\[\]\(\)\|:!\-]", " "))
        .withColumn(normalized_column_name, regexp_replace(col(normalized_column_name), r"\s+", " "))
        .withColumn(normalized_column_name, trim(col(normalized_column_name)))

        # Se o resultado ficar vazio, volta para o nome original em lowercase
        .withColumn(
            normalized_column_name,
            when(length(col(normalized_column_name)) > 0, col(normalized_column_name))
            .otherwise(lower(col(coluna_nome)))
        )
    )

In [14]:
df_old = criar_nome_normalizado(df_old,coluna_nome="Title")
#df_old.show()

In [15]:
df_new = criar_nome_normalizado(df_new,coluna_nome="Name")
#df_new.show()

## Correção colunas numericas

In [16]:
def correct_number_columns(col_name, num_type="bigint"):
    """
    Converts numeric string columns like:
    12K+ -> 12000
    1.5M -> 1500000
    2B   -> 2000000000
    950+ -> 950
    300  -> 300
    Handles commas in numbers.
    """

    c = upper(trim(col(col_name)))

    # Remove '+' wherever it appears, as it's not part of the number we want to cast
    c = regexp_replace(c, r"\+", "")

    # Remove commas from numbers (e.g., "483,372" -> "483372")
    c = regexp_replace(c, r",", "")

    # Extract only the numeric part and cast to DoubleType to handle decimals
    number = regexp_replace(c, r"[KMB]", "").cast(DoubleType())

    result = (
        when(c.rlike(r"K$"), number * 1000)
        .when(c.rlike(r"M$"), number * 1000000)
        .when(c.rlike(r"B$"), number * 1000000000)
        .otherwise(number)
        .cast(num_type) # Final cast to the desired integer type
    )

    return result

### Df

In [17]:
df_old = df_old.withColumn("Likes", correct_number_columns("Likes", "bigint"))
df_old = df_old.withColumn("Dislikes", correct_number_columns("Dislikes", "bigint"))
df_old = df_old.withColumn("Active", correct_number_columns("Active", "int"))
df_old = df_old.withColumn("Favourites", correct_number_columns("Favourites", "bigint"))
df_old = df_old.withColumn("Visits", correct_number_columns("Visits", "bigint"))

### Df_new

In [18]:
df_new=df_new.drop("Rank")
df_new=df_new.drop("Rating")

In [19]:
df_new=df_new.withColumn("Active",correct_number_columns("Active","int"))
df_new=df_new.withColumn("Visits",correct_number_columns("Visits","bigint"))
df_new=df_new.withColumn("Favourites",correct_number_columns("Favourites","bigint"))
df_new=df_new.withColumn("Likes",correct_number_columns("Likes","bigint"))
df_new=df_new.withColumn("Dislikes",correct_number_columns("Dislikes","bigint"))

In [20]:
cols_update = ["Visits", "Likes", "Dislikes", "Favourites", "Active"]

In [21]:
df_new_update = df_new.select("Normalized_name",
        *[col(c).alias(f"{c}_new")
        for c in cols_update
        if c in df_new.columns]
)

## Juntar dataframes

In [22]:
df_join = df_old.join(df_new_update, on="Normalized_name", how="left")

In [23]:
for c in cols_update:
    new_col = f"{c}_new"

    if c in df_join.columns and new_col in df_join.columns:
        df_join = df_join.withColumn(
            c,
            coalesce(col(new_col), col(c))
        )

In [24]:
df_final = df_join.drop(
    *[
        f"{c}_new"
        for c in cols_update
        if f"{c}_new" in df_join.columns
    ]
)

In [25]:
df_final.show()

+--------------------+--------------------+--------------------+-----------------+------+----------+----------+-------------+-------------+----------+----------+----------+---------------+-------+--------+--------------------+---------------+
|     Normalized_name|               Title|             Creator|AgeRecommendation|Active|Favourites|    Visits|    VoiceChat|       Camera|   Created|   Updated|ServerSize|          Genre|  Likes|Dislikes|            GameLink|    DateFetched|
+--------------------+--------------------+--------------------+-----------------+------+----------+----------+-------------+-------------+----------+----------+----------+---------------+-------+--------+--------------------+---------------+
|catalog avatar cr...|Catalog Avatar Cr...|         @ItsMuneeeb|         All Ages| 33714|   3421954|2850460040|    Supported|    Supported|  7/4/2021| 11/5/2024|        24|       Shopping|1390908|  123278|https://www.roblo...|11/5/2024 20:24|
|           the mimic| [HALL

In [26]:
df_final.printSchema()

root
 |-- Normalized_name: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Creator: string (nullable = true)
 |-- AgeRecommendation: string (nullable = true)
 |-- Active: integer (nullable = true)
 |-- Favourites: long (nullable = true)
 |-- Visits: long (nullable = true)
 |-- VoiceChat: string (nullable = true)
 |-- Camera: string (nullable = true)
 |-- Created: string (nullable = true)
 |-- Updated: string (nullable = true)
 |-- ServerSize: integer (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Likes: long (nullable = true)
 |-- Dislikes: long (nullable = true)
 |-- GameLink: string (nullable = true)
 |-- DateFetched: string (nullable = true)



# Analise inicial

## Ajustes

In [27]:
df=df_final

In [28]:
df.printSchema()

root
 |-- Normalized_name: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Creator: string (nullable = true)
 |-- AgeRecommendation: string (nullable = true)
 |-- Active: integer (nullable = true)
 |-- Favourites: long (nullable = true)
 |-- Visits: long (nullable = true)
 |-- VoiceChat: string (nullable = true)
 |-- Camera: string (nullable = true)
 |-- Created: string (nullable = true)
 |-- Updated: string (nullable = true)
 |-- ServerSize: integer (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Likes: long (nullable = true)
 |-- Dislikes: long (nullable = true)
 |-- GameLink: string (nullable = true)
 |-- DateFetched: string (nullable = true)



In [29]:
numeric_types = ["int", "bigint", "double", "float", "decimal", "long", "short"]

numeric_columns = [
    col_name
    for col_name, dtype in df.dtypes
    if dtype in numeric_types or dtype.startswith("decimal")
]
numeric_columns.append("Title")
numeric_columns.append("Genre")

In [31]:
df_num=df.select(numeric_columns)

In [35]:
df=df.pandas_api()

In [36]:
df_num=df_num.pandas_api()

## Analises